# Packages import

In [28]:
import os
import json
import requests
from bs4 import BeautifulSoup

# Ceneo Scraper

1. Provide url of the product's opinions page

In [24]:
product_code = "124893467"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send request to provided url

In [ ]:
response = requests.get(url)
print(response.status_code)

3. Fetch product name

In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [ ]:
product_name = page_dom.select_one('h1').get_text()
print(product_name)

4. Fetch all opinins from the webpage

In [ ]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")

5. Parse opinions to extract required data

In [19]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion.get("data-entry-id"),
        'author': opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recomendation > em').get_text().strip() if opinion.select_one('span.user-post__author-recomendation > em') else None,
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('div.user-post__text').get_text().strip(),
        'pros': [p.get_text().strip() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text().strip() for c in opinion.select('div.review-feature__item--negative')],
        'helpful': opinion.select_one('button.vote-yes > span').get_text().strip(),
        'unhelpful': opinion.select_one('button.vote-no > span').get_text().strip(),
        'publish_date': opinion.select_one('span.user-post__published > time:nth-child(1)').get('datetime').strip(),
        'purchase_date': opinion.select_one('span.user-post__published > time:nth-child(2)').get('datetime').strip() if opinion.select_one('span.user-post__published > time:nth-child(2)') else None,
    }
    all_opinions.append(single_opinion)

6. Check if there is next page with opinions

In [22]:
next = True if page_dom.select_one('button.pagination__next') else False
if next: page += 1

8. Save acquired opinions

In [29]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [35]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii=False)